### **Arquivos Brutos**

In [0]:
# ==============================================================================
# 0. IMPORTAÇÕES NECESSÁRIAS
# ==============================================================================
import json
import time
import pandas as pd
import requests

Dataset 1: Estatísticas do Pix

Quantidade e volume financeiro de transações Pix liquidadas mensalmente no período de novembro de 2020 a agosto de 2026.

Fonte: Bacen - https://dadosabertos.bcb.gov.br/


In [0]:
# ==============================================================================
# 1. CONFIGURAÇÕES E ENDPOINTS (API BCB OLINDA)
# ==============================================================================
CATALOG = "mvp"
SCHEMA = "staging"
VOLUME = "pix_transacoes"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

BASE_URL = (
    "https://olinda.bcb.gov.br/olinda/servico/Pix_DadosAbertos/versao/v1/odata/"
    "EstatisticasTransacoesPix(Database=@Database)"
)
COLS = (
    "AnoMes,PAG_PFPJ,REC_PFPJ,PAG_REGIAO,REC_REGIAO,PAG_IDADE,REC_IDADE,"
    "FORMAINICIACAO,NATUREZA,FINALIDADE,VALOR,QUANTIDADE"
)

# ==============================================================================
# 2. PROVISIONAMENTO DO VOLUME DE STAGING
# ==============================================================================
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"DROP VOLUME IF EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
spark.sql(f"CREATE VOLUME {CATALOG}.{SCHEMA}.{VOLUME}")

# ==============================================================================
# 3. EXTRAÇÃO HTTP PARA O VOLUME
# ==============================================================================
data_inicio = "2020-11-01"  # Entrada oficial em operação do Pix
data_fim = pd.Timestamp.now().strftime("%Y-%m-01")

periodos = pd.date_range(start=data_inicio, end=data_fim, freq="MS").strftime("%Y%m")
print(f"Total de competências no escopo: {len(periodos)} ({periodos[0]} até {periodos[-1]})")

total_arquivos_validos = 0
meses_vazios_consecutivos = 0

for periodo in periodos:
    print(f"📥 Requisitando competência {periodo} do BCB...")
    params = {
        "@Database": f"'{periodo}'",
        "$top": "10000",
        "$format": "json",
        "$select": COLS,
    }

    try:
        resp = requests.get(BASE_URL, params=params, timeout=60)

        if resp.status_code == 200:
            registros = resp.json().get("value", [])

            if len(registros) > 0:
                caminho_arquivo = f"{VOLUME_PATH}/pix_{periodo}.json"

                with open(caminho_arquivo, "w", encoding="utf-8") as f:
                    json.dump(registros, f)

                print(f"   ✅ Competência {periodo}: {len(registros)} registros salvos.")
                total_arquivos_validos += 1
                meses_vazios_consecutivos = 0
            else:
                print(f"   ⚠️ Competência {periodo}: Ainda não publicada pelo BCB.")
                meses_vazios_consecutivos += 1
        else:
            print(f"   ❌ Erro HTTP {resp.status_code} na competência {periodo}.")

    except requests.exceptions.RequestException as err:
        print(f"   ❌ Erro de conexão na competência {periodo}: {err}")

    # Circuit breaker: encerra após 3 meses seguidos sem publicação
    if meses_vazios_consecutivos >= 3:
        print("\n⏹️ Últimas 3 competências sem dados no BCB. Encerrando ingestão.")
        break

    time.sleep(0.1)

print(f"\n✨ Ingestão finalizada: {total_arquivos_validos} arquivo(s) gravado(s) em {VOLUME_PATH}.")

Total de competências no escopo: 71 (202011 até 202609)
📥 Requisitando competência 202011 do BCB...
   ✅ Competência 202011: 10000 registros salvos.
📥 Requisitando competência 202012 do BCB...
   ✅ Competência 202012: 10000 registros salvos.
📥 Requisitando competência 202101 do BCB...
   ✅ Competência 202101: 10000 registros salvos.
📥 Requisitando competência 202102 do BCB...
   ✅ Competência 202102: 10000 registros salvos.
📥 Requisitando competência 202103 do BCB...
   ✅ Competência 202103: 10000 registros salvos.
📥 Requisitando competência 202104 do BCB...
   ✅ Competência 202104: 10000 registros salvos.
📥 Requisitando competência 202105 do BCB...
   ✅ Competência 202105: 10000 registros salvos.
📥 Requisitando competência 202106 do BCB...
   ✅ Competência 202106: 10000 registros salvos.
📥 Requisitando competência 202107 do BCB...
   ✅ Competência 202107: 10000 registros salvos.
📥 Requisitando competência 202108 do BCB...
   ✅ Competência 202108: 10000 registros salvos.
📥 Requisitando

Dataset 2: Estatísticas de fraude no Pix (MED)

Reúne informações periódicas sobre transações Pix contestadas pelos usuários pagadores via Mecanismo Especial de Devolução (MED) previsto no Art. 41-B, inciso I, do Regulamento Anexo à Resolução BCB n° 1, de 12 de agosto de 2020, abrangendo tanto os volumes quanto os valores envolvidos em situações de suspeita de fraude.

Fonte: Bacen - https://dadosabertos.bcb.gov.br/

In [0]:
# ==============================================================================
# 1. PARÂMETROS DE AMBIENTE E API
# ==============================================================================
CATALOG = "mvp"
SCHEMA = "staging"
VOLUME = "pix_fraudes_med"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

BASE_URL = (
    "https://olinda.bcb.gov.br/olinda/servico/Pix_DadosAbertos/versao/v1/odata/"
    "EstatisticasFraudesPix(Database=@Database)"
)

COLS = (
    "AnoMes,"
    "QtdePixcontestados,"
    "Qtdecontestacoesaceitas,"
    "Qtdecontestacoesrejeitadas,"
    "Qtdecontestacoesaceitasacada100mil,"
    "QtdeUsuarioscommarcacoesdefraude,"
    "QtdeChavesPixcommarcacoesdefraude,"
    "ValorPixcontestadosaceitos,"
    "QuantidadedevolvidaintegralmentepormeiodoMED,"
    "ValorPixdevolvidosintegralmente,"
    "QuantidadedevolvidaparcialmentepormeiodoMED,"
    "ValorPixdevolvidosparcialmente,"
    "ValorPixresidualnaodevolvido,"
    "Quantidadedenaodevolvidossaldoinsuficiente,"
    "ValorPixnaodevolvidossaldoinsuficiente,"
    "Quantidadedenaodevolvidoscontaencerrada,"
    "Valornaodevolvidoscontaencerrada,"
    "Quantidadedenaodevolvidosmotivosdiversos,"
    "ValorPixnaodevolvidosmotivosdiversos,"
    "PercentualdeDevolucao,"
    "QtdePixbloqueadoscautelarmenteeliberados,"
    "ValorPixbloqueadoscautelarmenteeliberados,"
    "QtdePixbloqueadoscautelarmenteedevolvidos,"
    "ValorPixbloqueadoscautelarmenteedevolvidos"
)

# ==============================================================================
# 2. INFRAESTRUTURA DE STAGING
# ==============================================================================
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"DROP VOLUME IF EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
spark.sql(f"CREATE VOLUME {CATALOG}.{SCHEMA}.{VOLUME}")

# ==============================================================================
# 3. EXTRAÇÃO HTTP PARA O VOLUME
# ==============================================================================
data_inicio = "2021-11-01"  # Início oficial do MED
data_fim = pd.Timestamp.now().strftime("%Y-%m-01")

periodos = pd.date_range(start=data_inicio, end=data_fim, freq="MS").strftime("%Y%m")
print(f"Total de competências no escopo: {len(periodos)} ({periodos[0]} até {periodos[-1]})")

total_arquivos_validos = 0
meses_vazios_consecutivos = 0

for periodo in periodos:
    print(f"📥 Requisitando competência {periodo} do MED/BCB...")
    params = {
        "@Database": f"'{periodo}'",
        "$top": "10000",
        "$format": "json",
        "$select": COLS,
    }

    try:
        resp = requests.get(BASE_URL, params=params, timeout=60)

        if resp.status_code == 200:
            registros = resp.json().get("value", [])

            if len(registros) > 0:
                caminho_arquivo = f"{VOLUME_PATH}/pix_fraudes_{periodo}.json"

                with open(caminho_arquivo, "w", encoding="utf-8") as f:
                    json.dump(registros, f)

                print(f"   ✅ Competência {periodo}: {len(registros)} registros salvos.")
                total_arquivos_validos += 1
                meses_vazios_consecutivos = 0
            else:
                print(f"   ⚠️ Competência {periodo}: Retornou lista vazia no BCB.")
                meses_vazios_consecutivos += 1
        else:
            print(f"   ❌ Erro HTTP {resp.status_code} na competência {periodo}.")

    except requests.exceptions.RequestException as err:
        print(f"   ❌ Erro de conexão na competência {periodo}: {err}")

    # Encerra caso o endpoint legado não possua mais dados publicados
    if meses_vazios_consecutivos >= 3:
        print("\n⏹️ Últimas 3 competências sem dados no BCB. Encerrando ingestão.")
        break

    time.sleep(0.2)

print(f"\n✨ Ingestão concluída: {total_arquivos_validos} arquivo(s) gravado(s) em {VOLUME_PATH}.")

Total de competências no escopo: 59 (202111 até 202609)
📥 Requisitando competência 202111 do MED/BCB...
   ✅ Competência 202111: 52 registros salvos.
📥 Requisitando competência 202112 do MED/BCB...
   ✅ Competência 202112: 52 registros salvos.
📥 Requisitando competência 202201 do MED/BCB...
   ✅ Competência 202201: 52 registros salvos.
📥 Requisitando competência 202202 do MED/BCB...
   ✅ Competência 202202: 51 registros salvos.
📥 Requisitando competência 202203 do MED/BCB...
   ✅ Competência 202203: 50 registros salvos.
📥 Requisitando competência 202204 do MED/BCB...
   ✅ Competência 202204: 49 registros salvos.
📥 Requisitando competência 202205 do MED/BCB...
   ✅ Competência 202205: 48 registros salvos.
📥 Requisitando competência 202206 do MED/BCB...
   ✅ Competência 202206: 47 registros salvos.
📥 Requisitando competência 202207 do MED/BCB...
   ✅ Competência 202207: 46 registros salvos.
📥 Requisitando competência 202208 do MED/BCB...
   ✅ Competência 202208: 45 registros salvos.
📥 Re

Dataset 3: Censo Demográfico IBGE por regiões

Dados demográficos da população brasileira por regiões (Censo 2022)

Fonte: API SIDRA IBGE - https://apisidra.ibge.gov.br/values/t/4714/n2/all/v/93/p/last%201/f/a

In [0]:
# ==============================================================================
# 1. CONFIGURAÇÕES DE CATÁLOGO, SCHEMA E VOLUME
# ==============================================================================
CATALOG = "mvp"
SCHEMA = "staging"
VOLUME = "ibge_populacao"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

# Tabela 4714 (Censo Demográfico) - População residente (v=93) por Grande Região (n2/all)
BASE_URL_SIDRA = "https://apisidra.ibge.gov.br/values/t/4714/n2/all/v/93/p/last%201/f/a"

# ==============================================================================
# 2. INFRAESTRUTURA DE STAGING
# ==============================================================================
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"DROP VOLUME IF EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
spark.sql(f"CREATE VOLUME {CATALOG}.{SCHEMA}.{VOLUME}")

# ==============================================================================
# 3. EXTRAÇÃO HTTP PARA O VOLUME
# ==============================================================================
print("📥 Requisitando dados demográficos da API SIDRA/IBGE...")
resp = requests.get(BASE_URL_SIDRA, timeout=60)
raw_data = resp.json() if resp.status_code == 200 else []

# O primeiro elemento da API SIDRA traz apenas cabeçalhos/descrições de colunas
dados = raw_data[1:] if len(raw_data) > 1 else []

if dados:
    caminho_arquivo = f"{VOLUME_PATH}/ibge_populacao_regioes.json"
    with open(caminho_arquivo, "w", encoding="utf-8") as f:
        json.dump(dados, f, ensure_ascii=False)
    print(f"✅ Salvo no Volume: {caminho_arquivo} ({len(dados)} regiões)")
else:
    print("❌ Falha na coleta dos dados do IBGE.")

📥 Requisitando dados demográficos da API SIDRA/IBGE...
✅ Salvo no Volume: /Volumes/mvp/staging/ibge_populacao/ibge_populacao_regioes.json (5 regiões)
